In [1]:
import os
import re
import json
import shutil
import torch
import numpy as np
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,   
    set_seed                 
)
from sklearn.metrics import accuracy_score, f1_score, classification_report
from IPython.display import FileLink
EMOTICON_MODE = "map"

EMOTICON_MAP = {
    ":)))": " rất vui ", ":))": " vui vẻ ", ":)": " vui ",
    "=)))": " cười lớn ", "=))": " cười vui ", ":v": " vui vẻ ",
    "<3": " yêu thích ", ":(((": " rất buồn ", ":((": " buồn ", ":(": " buồn "
}

def clean_text(text):
    text = str(text).strip()
    text = re.sub(r"http\S+|www\.\S+", " ", text) 
    text = re.sub(r"@\w+", " ", text)          
    for emo, meaning in EMOTICON_MAP.items():
        text = text.replace(emo, meaning)
    text = re.sub(r"(?!\d)(.)\1{2,}", r"\1\1", text)
    return re.sub(r"\s+", " ", text).strip()

print("Cell 1: Đã nạp thư viện và hàm tiền xử lý thành công!")

Cell 1: Đã nạp thư viện và hàm tiền xử lý thành công!


In [2]:
VSMEC_REPEAT = 2   # VSFC lớn gấp ~2 lần VSMEC nên nhân VSMEC để cân bằng miền; đặt 1 để không nhân
 
print("1. Đang tải bộ dữ liệu Mạng xã hội (UIT-VSMEC)...")
ds_social = load_dataset("tridm/UIT-VSMEC")
 
def map_social_labels(example):
    emo = example["Emotion"]
    if emo in ["Anger", "Disgust", "Sadness", "Fear"]:
        label = 0  # Tiêu cực
    elif emo in ["Other", "Surprise"]:
        label = 1  # Trung tính
    else:
        label = 2  # Tích cực (Enjoyment)
    return {"text": clean_text(example["Sentence"]), "label": label}
 
ds_social = ds_social.map(map_social_labels, remove_columns=ds_social["train"].column_names)
 
print("2. Đang tải bộ dữ liệu Đánh giá mở rộng (UIT-VSFC)...")
ds_review = load_dataset("uitnlp/vietnamese_students_feedback", revision="refs/convert/parquet")
 
def map_review_labels(example):
    return {"text": clean_text(example["sentence"]), "label": int(example["sentiment"])}
 
ds_review = ds_review.map(map_review_labels, remove_columns=ds_review["train"].column_names)
 
train_dataset = concatenate_datasets([ds_social["train"]] * VSMEC_REPEAT + [ds_review["train"]]).shuffle(seed=42)
val_dataset   = concatenate_datasets([ds_social["validation"], ds_review["validation"]])
test_dataset  = concatenate_datasets([ds_social["test"], ds_review["test"]])
 
print(f"Cell 2: VSMEC lặp {VSMEC_REPEAT} lần | Tổng số câu Huấn luyện: {len(train_dataset)} | "
      f"Validation: {len(val_dataset)} | Test: {len(test_dataset)}")
 

1. Đang tải bộ dữ liệu Mạng xã hội (UIT-VSMEC)...


README.md:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

train.json:   0%|          | 0.00/777k [00:00<?, ?B/s]

valid.json:   0%|          | 0.00/96.6k [00:00<?, ?B/s]

test.json:   0%|          | 0.00/96.4k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5548 [00:00<?, ? examples/s]

... (15 outputs left out)

Map:   0%|          | 0/11426 [00:00<?, ? examples/s]

Map:   0%|          | 0/1583 [00:00<?, ? examples/s]

Map:   0%|          | 0/3166 [00:00<?, ? examples/s]

Cell 2: VSMEC lặp 2 lần | Tổng số câu Huấn luyện: 22522 | Validation: 2269 | Test: 3859


In [3]:
MODEL_NAME = "uitnlp/visobert"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

print("Đang mã hóa (Tokenize) toàn bộ dữ liệu...")
tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_val   = val_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_test  = test_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Cell 3: Đã mã hóa dữ liệu hoàn tất!")

config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B /  471kB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

Đang mã hóa (Tokenize) toàn bộ dữ liệu...


Map:   0%|          | 0/22522 [00:00<?, ? examples/s]

Map:   0%|          | 0/2269 [00:00<?, ? examples/s]

Map:   0%|          | 0/3859 [00:00<?, ? examples/s]

Cell 3: Đã mã hóa dữ liệu hoàn tất!


In [4]:
ID2LABEL = {0: "Tiêu cực", 1: "Trung tính", 2: "Tích cực"}
LABEL2ID = {"Tiêu cực": 0, "Trung tính": 1, "Tích cực": 2}

set_seed(42)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1_macro": f1_macro}

print("Cell 4: Đã khởi tạo mô hình ViSoBERT(3 nhãn)!")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  390MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: uitnlp/visobert
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Cell 4: Đã khởi tạo mô hình ViSoBERT(3 nhãn)!


In [6]:
import math

BATCH_SIZE = 64
EPOCHS = 5
n_gpu = max(1, torch.cuda.device_count())

steps_per_epoch = math.ceil(len(tokenized_train) / (BATCH_SIZE * n_gpu))
warmup = int(0.1 * steps_per_epoch * EPOCHS)   
print(f"Số bước mỗi epoch: {steps_per_epoch} | Warmup: {warmup} bước")
print(f"4.Bắt đầu huấn luyện trên {torch.cuda.device_count()} GPU...")

training_args = TrainingArguments(
    output_dir="./checkpoints_visobert",
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,   
    per_device_eval_batch_size=128,
    num_train_epochs=EPOCHS,                  
    warmup_steps=warmup,                      
    weight_decay=0.01,
    label_smoothing_factor=0.05,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=torch.cuda.is_available(),
    report_to="none"
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()
print("Cell 5: Huấn luyện hoàn tất!")

Số bước mỗi epoch: 176 | Warmup: 88 bước
4.Bắt đầu huấn luyện trên 2 GPU...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,1.023801,0.843984,0.740629
2,No log,0.986717,0.856324,0.760269
3,1.041720,1.052359,0.853680,0.751127
4,1.041720,1.125027,0.855443,0.752534


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Cell 5: Huấn luyện hoàn tất!


In [ ]:
# print("KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST (chưa cộng b):")
# test_output = trainer.predict(tokenized_test)
# test_preds = np.argmax(test_output.predictions, axis=-1)
# print(classification_report(tokenized_test["label"], test_preds,
#                             target_names=["Tiêu cực", "Trung tính", "Tích cực"], digits=4))
 
# SAVE_DIR = "./Model_viso_bert_v2"
# os.makedirs(SAVE_DIR, exist_ok=True)
 
# trainer.save_model(SAVE_DIR)           # đã là model tốt nhất nhờ load_best_model_at_end
# tokenizer.save_pretrained(SAVE_DIR)
# print("Đã lưu model và tokenizer. Cấu hình suy luận (b, T, ngưỡng) sẽ được ghi ở Cell F "
#       "sau khi chọn trên validation.")

In [7]:

print("KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST:")
test_output = trainer.predict(tokenized_test)
test_preds = np.argmax(test_output.predictions, axis=-1)
print(classification_report(tokenized_test["label"], test_preds, target_names=["Tiêu cực", "Trung tính", "Tích cực"], digits=4))

SAVE_DIR = "./Model_viso_bert_v2"
os.makedirs(SAVE_DIR, exist_ok=True)

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

inference_cfg = {
    "temperature": 1.3,
    "emoji_mode": "keep",
    "max_len": 128,
    "labels": ["neg", "neu", "pos"]
}
with open(os.path.join(SAVE_DIR, "inference_config.json"), "w", encoding="utf-8") as f:
    json.dump(inference_cfg, f, ensure_ascii=False, indent=2)

shutil.make_archive("Model_viso_bert_v2", "zip", SAVE_DIR)
print("\nĐÃ ĐÓNG GÓI XONG 5 FILE! Hãy bấm vào dòng link màu xanh bên dưới để tải về máy:")
FileLink(r"Model_viso_bert_v2.zip")

KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST:


              precision    recall  f1-score   support

    Tiêu cực     0.8667    0.9254    0.8951      1743
  Trung tính     0.5918    0.3483    0.4386       333
    Tích cực     0.8929    0.9024    0.8976      1783

    accuracy                         0.8650      3859
   macro avg     0.7838    0.7254    0.7438      3859
weighted avg     0.8551    0.8650    0.8569      3859



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


ĐÃ ĐÓNG GÓI XONG 5 FILE! Hãy bấm vào dòng link màu xanh bên dưới để tải về máy:


/kaggle/working/Model_viso_bert_v2.zip

TEST

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Nạp lại mô hình vừa lưu từ thư mục ./Model_viso_bert_v2
MODEL_DIR = "./Model_viso_bert_v2"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_tok = AutoTokenizer.from_pretrained(MODEL_DIR)
test_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
test_model.eval()

def test_predict(comments, T=1.3):
    print(f"{'BÌNH LUẬN':<52} | {'KẾT LUẬN':<12} | {'ĐIỂM':<6} | {'CHI TIẾT XÁC SUẤT'}")
    print("-" * 105)
    
    scores = []
    counts = {"Tích cực": 0, "Trung tính": 0, "Hỗn hợp": 0, "Tiêu cực": 0}
    
    for raw in comments:
        cleaned = clean_text(raw) or "..."
        inputs = test_tok(cleaned, return_tensors="pt", truncation=True, max_length=128).to(device)
        
        with torch.no_grad():
            logits = test_model(**inputs).logits
            probs = F.softmax(logits.float() / T, dim=-1).squeeze().cpu().numpy()
            
        neg_p, neu_p, pos_p = float(probs[0]), float(probs[1]), float(probs[2])
        score_100 = round(50.0 + 50.0 * (pos_p - neg_p), 1)
        scores.append(score_100)
        
        # Quy tắc phân nhóm đồng bộ với app.py
        if min(neg_p, pos_p) >= 0.25:
            label = "🟣 Hỗn hợp"
            counts["Hỗn hợp"] += 1
        elif neu_p >= 0.50:
            label = "⚪ Trung tính"
            counts["Trung tính"] += 1
        elif score_100 >= 60:
            label = "🟢 Tích cực"
            counts["Tích cực"] += 1
        elif score_100 > 40:
            label = "⚪ Trung tính"
            counts["Trung tính"] += 1
        else:
            label = "🔴 Tiêu cực"
            counts["Tiêu cực"] += 1
            
        prob_str = f"Neg:{neg_p*100:4.1f}% | Neu:{neu_p*100:4.1f}% | Pos:{pos_p*100:4.1f}%"
        short_text = (raw[:49] + "...") if len(raw) > 52 else raw
        print(f"{short_text:<52} | {label:<12} | {score_100:5.1f}đ | {prob_str}")
        
    print("-" * 105)
    print(f"📊 Điểm trung bình bài đăng: {sum(scores)/len(scores):.1f}/100")
    print(f"📌 Thống kê: Tích cực: {counts['Tích cực']} | Trung tính: {counts['Trung tính']} | Hỗn hợp: {counts['Hỗn hợp']} | Tiêu cực: {counts['Tiêu cực']}")

# 2. Bộ câu test cho các tình huống (Khen, Chê, Hỏi han, Câu đối lập "nhưng")
# sample_comments = [
#     "Hàng dùng mượt phết, shop phục vụ nhiệt tình",
#     "Giao diện nhìn hơi phèn cơ mà dùng mượt phết =))",
#     "Quảng cáo thì rầm rộ đấy nhưng chất lượng quá tệ",
#     "Tưởng chán nhưng xem cuốn thực sự luôn :)))",
#     "Món ăn trình bày đẹp nhưng hơi nguội và mặn",
#     "Cho mình hỏi shop ở quận mấy Hà Nội vậy ạ?",
#     "Bài viết bình thường, chưa có gì đặc sắc",
#     "Ship chậm quá, gọi điện không thèm nghe máy :((",
#     "Tuy hơi buồn ngủ nhưng hôm nay rất là vui",
#     "Video chán quá xem 200 lần rồi chưa thấy hay",
#     "Slide dễ hiểu nhưng kiến thức nặng quá",
#     "Bài làm khá ổn rồi, bạn nên sửa thêm giao diện nữa là đẹp",
#     "Bài làm khá ổn nhưng giao diện thì xấu",
#     "Buổi giảng rất hay nhưng sinh viên ngủ gật và không nghe giảng",
#     "Tuy buổi dạy làm tôi buồn ngủ nhưng kiến thức rất hay"
# ]
sample_comments = [
    "Hàng đẹp tuyệt vời, giao nhanh như chớp, mỗi tội gửi nhầm mẫu mình đặt :)))",
    "Quán nấu ăn rất có tâm, giúp mình cai nghiện thành công món bún bò luôn",
    "Mua về làm cục chặn giấy thì hợp chứ nghe nhạc gì tầm này",
    "Phim kinh dị làm tốt thật sự, xem lúc nửa đêm mà sợ phát khóc luôn",
    "Chưa dùng thử nên chưa biết bền không, tạm cho 5 sao vì đóng gói kỹ",
    "Cũng không phải là không ngon, nhưng bảo quay lại lần hai thì chắc chắn không",
    "Sản phẩm đúng mô tả, dùng ổn áp trong tầm giá, shop tư vấn nhiệt tình ",
    "Thái độ nhân viên lồi lõm thật sự, lần sau có cho tiền cũng chẳng dám ghé",
    "Tưởng hàng lởm mà hóa ra cũng lởm thật =))",
    "Cứ tưởng sẽ thất vọng như mọi khi ai ngờ lần này shop làm mình bất ngờ thật"
]
test_predict(sample_comments)

In [ ]:
# %% CELL A — Hàm dùng chung (cell này không train gì cả)
import os, re, json, math, gc, time, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (f1_score, accuracy_score, classification_report,
                             confusion_matrix, cohen_kappa_score)
 
NAMES = ["Tiêu cực", "Trung tính", "Tích cực"]
 
 
def softmax(z, T=1.0):
    """Softmax có nhiệt độ T (T > 1 làm xác suất 'phẳng' hơn)."""
    z = np.asarray(z, dtype=np.float64) / T
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)
 
 
def get_logits(ds, tr=None):
    """ds có cột 'text' và 'label'. Trả về (logits, nhãn thật, danh sách văn bản)."""
    tr = tr if tr is not None else trainer
    tok = ds.map(tokenize_function, batched=True, remove_columns=["text"])
    out = tr.predict(tok)
    return out.predictions, np.array(ds["label"]), list(ds["text"])
 
 
def apply_bias(logits, b):
    """Cộng độ lệch b vào điểm (logit) của lớp Trung tính (cột 1)."""
    z = np.array(logits, dtype=np.float64, copy=True)
    z[:, 1] += b
    return z
 
 
def macro_f1(labels, logits, b=0.0):
    return f1_score(labels, apply_bias(logits, b).argmax(-1), average="macro")
 
 
def fit_bias(logits, labels, grid=np.arange(0, 3.01, 0.25)):
    """Chọn b tốt nhất theo macro F1. Chỉ gọi với dữ liệu VALIDATION."""
    scores = {round(float(b), 2): macro_f1(labels, logits, b) for b in grid}
    best_b = max(scores, key=scores.get)
    return best_b, scores
 
 
def nll(logits, labels, T=1.0):
    """Cross-entropy trung bình (càng nhỏ càng tốt)."""
    p = softmax(logits, T)
    return float(-np.log(p[np.arange(len(labels)), labels] + 1e-12).mean())
 
 
def fit_temperature(logits, labels, grid=np.arange(0.5, 3.01, 0.05)):
    """Chọn nhiệt độ T làm NLL nhỏ nhất. Chỉ gọi với dữ liệu VALIDATION."""
    best = min(grid, key=lambda T: nll(logits, labels, T))
    return round(float(best), 2)
 
 
def ece(probs, labels, n_bins=10):
    """Expected Calibration Error: độ lệch giữa 'độ tự tin' và độ chính xác thật."""
    conf = probs.max(-1)
    correct = probs.argmax(-1) == labels
    bins = np.linspace(0, 1, n_bins + 1)
    total = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(total)
 
 
def coverage_table(probs, labels, thresholds):
    """Với mỗi ngưỡng: giữ lại bao nhiêu % câu và accuracy trên phần giữ lại."""
    conf = probs.max(-1)
    correct = probs.argmax(-1) == labels
    rows = []
    for t in thresholds:
        keep = conf >= t
        rows.append({
            "ngưỡng": float(t),
            "tỉ lệ câu được giữ": float(keep.mean()),
            "accuracy trên phần giữ": float(correct[keep].mean()) if keep.any() else float("nan"),
        })
    return pd.DataFrame(rows)
 
 
def chuan_hoa(t):
    return re.sub(r"\s+", " ", str(t).lower()).strip()
 
 
def overlap_report(train_texts, other_texts, name):
    """Đếm số câu của tập khác đã xuất hiện nguyên văn trong train."""
    tr = set(chuan_hoa(t) for t in train_texts)
    dup = [i for i, t in enumerate(other_texts) if chuan_hoa(t) in tr]
    print(f"{name}: {len(dup)}/{len(other_texts)} câu "
          f"({len(dup) / max(1, len(other_texts)):.1%}) đã có trong train")
    return dup
 
 
def label_conflicts(texts, labels):
    """Những câu giống hệt nhau nhưng bị gán nhãn khác nhau."""
    seen = {}
    for t, l in zip(texts, labels):
        seen.setdefault(chuan_hoa(t), set()).add(int(l))
    return {t: ls for t, ls in seen.items() if len(ls) > 1}
 
 
print("Cell A: đã nạp hàm dùng chung")
 
 
# %% CELL B — Lấy dự đoán của model hiện tại + kiểm tra trùng lặp train/test
CACHE = {}
CACHE["VSMEC"] = get_logits(ds_social["test"])
CACHE["VSFC"] = get_logits(ds_review["test"])
CACHE["GOP"] = (                                   # thứ tự giống test_dataset: VSMEC rồi VSFC
    np.concatenate([CACHE["VSMEC"][0], CACHE["VSFC"][0]]),
    np.concatenate([CACHE["VSMEC"][1], CACHE["VSFC"][1]]),
    CACHE["VSMEC"][2] + CACHE["VSFC"][2],
)
CACHE_VAL = get_logits(val_dataset)
print("Test gộp:", len(CACHE["GOP"][1]), "câu | Validation:", len(CACHE_VAL[1]), "câu")
 
# Bản train GỐC (không nhân đôi VSMEC) để kiểm tra trùng lặp
# (datasets bản mới trả về 'Column' chứ không phải list nên phải bọc list() trước khi cộng)
train_goc = list(ds_social["train"]["text"]) + list(ds_review["train"]["text"])
train_goc_labels = list(ds_social["train"]["label"]) + list(ds_review["train"]["label"])
 
xung_dot = label_conflicts(train_goc, train_goc_labels)
print(f"\nCâu giống nhau nhưng khác nhãn trong train: {len(xung_dot)}")
for t, ls in list(xung_dot.items())[:5]:
    print("   ", sorted(ls), "|", t[:80])
 
print()
for name in ("VSMEC", "VSFC"):
    logits, labels, texts = CACHE[name]
    dup = set(overlap_report(train_goc, texts, f"Test {name}"))
    if dup:
        keep = np.array([i not in dup for i in range(len(texts))])
        print(f"   macro F1: toàn bộ {macro_f1(labels, logits):.4f} | "
              f"chỉ câu KHÔNG trùng {macro_f1(labels[keep], logits[keep]):.4f}")
overlap_report(train_goc, CACHE_VAL[2], "Validation")
print("\nNếu tỉ lệ trùng rất nhỏ thì điểm test đáng tin; nếu lớn, hãy báo cáo điểm 'chỉ câu không trùng'.")
 
 
# %% CELL C — Chọn b, T và ngưỡng "không chắc chắn" TRÊN VALIDATION
val_logits, val_labels, _ = CACHE_VAL
 
best_b, b_scores = fit_bias(val_logits, val_labels)
print("val macro F1 theo b:", {k: round(v, 4) for k, v in b_scores.items()})
print(f"-> b tốt nhất = {best_b}")
 
z_val = apply_bias(val_logits, best_b)
TEMP = fit_temperature(z_val, val_labels)
print(f"-> nhiệt độ T tốt nhất = {TEMP}   (thay cho con số 1.3 tự đặt)")
if TEMP in (0.5, 3.0):
    print("   Cảnh báo: T nằm sát biên của khoảng tìm kiếm (0.5 - 3.0), kết quả có thể chưa tối ưu.")
 
# Xác suất có đáng tin hơn không? (ECE và NLL càng nhỏ càng tốt)
print()
for name, (lg, lb, _) in [("Validation", CACHE_VAL), ("Test gộp", CACHE["GOP"])]:
    print(f"{name}: ECE {ece(softmax(lg), lb):.4f} -> {ece(softmax(apply_bias(lg, best_b), TEMP), lb):.4f}"
          f" | NLL {nll(lg, lb):.4f} -> {nll(apply_bias(lg, best_b), lb, TEMP):.4f}"
          f"   (trước -> sau khi cộng b và chia T)")
 
# Ngưỡng "không chắc chắn": thay cho các luật 0.25 / 0.5 / 60 / 40 tự đặt
probs_val = softmax(z_val, TEMP)
tbl = coverage_table(probs_val, val_labels, np.round(np.arange(0.40, 0.96, 0.05), 2))
print("\nBảng trên VALIDATION (giữ lại các câu có độ chắc chắn >= ngưỡng):")
print(tbl.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
 
TARGET_ACC = 0.90      # mục tiêu: các câu 'được giữ' phải đúng ít nhất 90%
ok = tbl[tbl["accuracy trên phần giữ"] >= TARGET_ACC]
if len(ok):
    CONF_THRESHOLD = float(ok["ngưỡng"].iloc[0])
    print(f"\n-> CONF_THRESHOLD = {CONF_THRESHOLD}  (ngưỡng nhỏ nhất đạt accuracy >= {TARGET_ACC} trên validation)")
    if CONF_THRESHOLD <= 0.4:
        print("   (model đã đạt mục tiêu mà không cần loại câu nào, nên ngưỡng này gần như không có tác dụng)")
else:
    CONF_THRESHOLD = 0.5
    print(f"\n-> Không có ngưỡng nào đạt accuracy >= {TARGET_ACC}; tạm đặt CONF_THRESHOLD = 0.5. "
          f"Hãy tự chọn một ngưỡng từ bảng trên (sửa biến này) rồi chạy tiếp.")
 
# Kiểm tra lại trên TEST (không dùng test để chọn ngưỡng)
lg, lb, _ = CACHE["GOP"]
p_te = softmax(apply_bias(lg, best_b), TEMP)
keep = p_te.max(-1) >= CONF_THRESHOLD
if keep.any():
    print(f"Test gộp: giữ {keep.mean():.1%} câu, accuracy phần giữ = "
          f"{(p_te.argmax(-1)[keep] == lb[keep]).mean():.4f} | toàn bộ = {(p_te.argmax(-1) == lb).mean():.4f}")
 
# Điểm macro F1 từng tập test, trước và sau khi cộng b
rows = []
for name in ("VSMEC", "VSFC", "GOP"):
    lg, lb, _ = CACHE[name]
    rows.append({"Tập test": name, "macro F1 (b=0)": macro_f1(lb, lg),
                 "macro F1 (có b)": macro_f1(lb, lg, best_b)})
print("\n" + pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
 
 

In [ ]:
# %% CELL D — Phân tích lỗi: ma trận nhầm lẫn + các câu model sai
def phan_tich_loi(name, b=0.0, n_show=15):
    logits, labels, texts = CACHE[name]
    z = apply_bias(logits, b)
    pred = z.argmax(-1)
    probs = softmax(z)
 
    cm = confusion_matrix(labels, pred, labels=[0, 1, 2], normalize="true")
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(NAMES, rotation=20); ax.set_yticklabels(NAMES)
    ax.set_xlabel("Dự đoán"); ax.set_ylabel("Thực tế")
    ax.set_title(f"Ma trận nhầm lẫn (theo hàng) - {name}", fontsize=10)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center",
                    color="white" if cm[i, j] > 0.5 else "black")
    plt.tight_layout()
    plt.savefig(f"confusion_{name}.png", dpi=150)
    plt.show()
 
    sai_tn = [i for i in range(len(texts)) if labels[i] == 1 and pred[i] != 1]
    print(f"\n--- {name}: {len(sai_tn)} câu Trung tính bị đoán sai (hiện {n_show} câu đầu) ---")
    for i in sai_tn[:n_show]:
        print(f"[đoán {NAMES[pred[i]]} {probs[i, pred[i]]:.2f}] {texts[i]}")
 
    sai_tin = [i for i in range(len(texts)) if pred[i] != labels[i] and probs[i, pred[i]] >= 0.9]
    print(f"\n--- {name}: {len(sai_tin)} câu đoán SAI nhưng rất tự tin (>= 0.9). "
          f"Đọc xem nhãn gốc có hợp lý không ---")
    for i in sai_tin[:10]:
        print(f"[thật: {NAMES[labels[i]]} | đoán: {NAMES[pred[i]]} {probs[i, pred[i]]:.2f}] {texts[i]}")
 
 
phan_tich_loi("VSFC", b=best_b)
phan_tich_loi("VSMEC", b=best_b)
 
 
# %% CELL E — Baseline TF-IDF + Logistic Regression và bảng so sánh
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
 
# Baseline học trên đúng train_dataset mà ViSoBERT dùng (có VSMEC x2).
# Lưu ý: baseline dùng class_weight="balanced" nên được lợi ở lớp Trung tính.
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)
X_tr = tfidf.fit_transform(list(train_dataset["text"]))
base = LogisticRegression(max_iter=2000, class_weight="balanced").fit(X_tr, list(train_dataset["label"]))
 
 
def dong(ten_tap, ten_model, labels, pred):
    f = f1_score(labels, pred, average=None, labels=[0, 1, 2])
    return {"Tập test": ten_tap, "Mô hình": ten_model,
            "accuracy": accuracy_score(labels, pred),
            "macro F1": f1_score(labels, pred, average="macro"),
            "F1 Trung tính": f[1]}
 
 
rows = []
for name in ("VSMEC", "VSFC", "GOP"):
    lg, lb, tx = CACHE[name]
    rows.append(dong(name, "TF-IDF + LogReg", lb, base.predict(tfidf.transform(tx))))
    rows.append(dong(name, "ViSoBERT (b=0)", lb, lg.argmax(-1)))
    rows.append(dong(name, f"ViSoBERT (b={best_b})", lb, apply_bias(lg, best_b).argmax(-1)))
bang = pd.DataFrame(rows)
print(bang.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
bang.to_csv("bang_so_sanh_baseline.csv", index=False)
 
 
# %% CELL F — Lưu config thống nhất + demo dùng ĐÚNG luật đã đánh giá
import torch
from IPython.display import FileLink, display
from transformers import AutoTokenizer, AutoModelForSequenceClassification
 
SAVE_DIR = globals().get("SAVE_DIR", "./Model_viso_bert_v2")
if not os.path.isdir(SAVE_DIR):               # chưa lưu thì lưu model tốt nhất hiện có
    os.makedirs(SAVE_DIR, exist_ok=True)
    trainer.save_model(SAVE_DIR)
    tokenizer.save_pretrained(SAVE_DIR)
 
inference_cfg = {
    "temperature": TEMP,                      # chọn trên validation
    "neutral_bias": best_b,                   # chọn trên validation
    "confidence_threshold": CONF_THRESHOLD,   # chọn trên validation
    "emoji_mode": "keep",
    "max_len": 128,
    "labels": ["neg", "neu", "pos"],
    "note": "Văn bản phải qua clean_text() giống lúc huấn luyện. "
            "Điểm 0-100 = 50 + 50*(P_pos - P_neg) là điểm tóm tắt, chưa được kiểm chứng riêng.",
}
with open(os.path.join(SAVE_DIR, "inference_config.json"), "w", encoding="utf-8") as f:
    json.dump(inference_cfg, f, ensure_ascii=False, indent=2)
shutil.make_archive("Model_viso_bert_v2", "zip", SAVE_DIR)
print("Đã cập nhật inference_config.json và nén lại thành Model_viso_bert_v2.zip")
display(FileLink("Model_viso_bert_v2.zip"))
 
# Nạp lại model từ thư mục đã lưu để chắc chắn bản đóng gói chạy được
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tok_d = AutoTokenizer.from_pretrained(SAVE_DIR)
mdl_d = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to(device).eval()
with open(os.path.join(SAVE_DIR, "inference_config.json"), encoding="utf-8") as f:
    CFG = json.load(f)
 
 
def du_doan(comments):
    rows = []
    for raw in comments:
        enc = tok_d(clean_text(raw) or "...", return_tensors="pt",
                    truncation=True, max_length=CFG["max_len"]).to(device)
        with torch.no_grad():
            z = mdl_d(**enc).logits[0].float().cpu().numpy()
        z[1] += CFG["neutral_bias"]
        p = softmax(z[None], CFG["temperature"])[0]
        k = int(p.argmax())
        rows.append({
            "Bình luận": raw if len(raw) <= 60 else raw[:57] + "...",
            "Nhãn": NAMES[k],
            "Độ chắc chắn": round(float(p[k]), 3),
            "Điểm 0-100": round(50 + 50 * float(p[2] - p[0]), 1),
            "Ghi chú": "⚠ không chắc chắn" if p[k] < CFG["confidence_threshold"] else "",
        })
    return pd.DataFrame(rows)
 
 
mau = globals().get("sample_comments", ["Hàng dùng mượt phết, shop phục vụ nhiệt tình",
                                        "Cho mình hỏi shop ở quận mấy Hà Nội vậy ạ?",
                                        "Ship chậm quá, gọi điện không thèm nghe máy :(("])
print(du_doan(mau).to_string(index=False))
# Lưu ý: hãy dùng đúng luật này (b, T, ngưỡng) trong app.py để demo khớp với số liệu báo cáo.
 

In [ ]:
# %% CELL G — Đánh giá trên BÌNH LUẬN TỰ THU của bạn (bằng chứng quan trọng nhất)
# CÁCH LÀM:
#   1) Chạy cell này một lần: nó tạo file mẫu binh_luan_mau.csv.
#   2) Thu 100-200 bình luận CÔNG KHAI của đúng lĩnh vực đề tài. Chỉ giữ phần chữ,
#      XOÁ tên người dùng, số điện thoại, link. Ít nhất ~30 câu mỗi lớp (cả Trung tính).
#   3) Gán nhãn tay: neg / neu / pos (cột label). Nếu có người thứ hai gán độc lập,
#      điền cột label2 để tính độ đồng thuận (Cohen's kappa).
#   4) Lưu thành binh_luan_cua_toi.csv (UTF-8), upload lên Kaggle rồi đổi MY_CSV bên dưới.
from datasets import Dataset
 
MY_CSV = "binh_luan_cua_toi.csv"      # nếu upload dạng Dataset: "/kaggle/input/<ten>/binh_luan_cua_toi.csv"
 
if not os.path.exists(MY_CSV):
    pd.DataFrame({
        "text": ["(dán bình luận 1 vào đây)", "(dán bình luận 2 vào đây)"],
        "label": ["pos", "neg"],      # chỉ nhận: neg / neu / pos
        "label2": ["", ""],           # (tuỳ chọn) nhãn của người gán thứ hai
    }).to_csv("binh_luan_mau.csv", index=False, encoding="utf-8-sig")
    print("Chưa có", MY_CSV, "-> đã tạo binh_luan_mau.csv để bạn làm theo mẫu. Làm xong thì chạy lại cell này.")
else:
    mine = pd.read_csv(MY_CSV).dropna(subset=["text", "label"]).copy()
    lab_map = {"neg": 0, "neu": 1, "pos": 2}
    mine["y"] = mine["label"].astype(str).str.strip().str.lower().map(lab_map)
    if mine["y"].isna().any():
        print("Cảnh báo: bỏ", int(mine["y"].isna().sum()), "dòng có nhãn không hợp lệ (chỉ nhận neg/neu/pos)")
        mine = mine.dropna(subset=["y"])
    print("Số câu theo nhãn:", mine["y"].map({0: "neg", 1: "neu", 2: "pos"}).value_counts().to_dict())
    if mine["y"].nunique() < 3 or mine["y"].value_counts().min() < 30:
        print("Lưu ý: có lớp dưới 30 câu (hoặc thiếu lớp), F1 của lớp đó sẽ dao động rất mạnh.")
 
    ds_mine = Dataset.from_dict({"text": [clean_text(t) for t in mine["text"]],
                                 "label": mine["y"].astype(int).tolist()})
    lg, lb, tx = get_logits(ds_mine)
    pred = apply_bias(lg, best_b).argmax(-1)
    print(f"\n=== ViSoBERT (b={best_b}) trên bình luận tự thu ===")
    print(classification_report(lb, pred, labels=[0, 1, 2], target_names=NAMES, digits=4, zero_division=0))
    print("macro F1 khi b=0:", round(macro_f1(lb, lg), 4), "| khi có b:", round(macro_f1(lb, lg, best_b), 4))
    if "base" in globals():
        pb = base.predict(tfidf.transform(tx))
        print("macro F1 baseline TF-IDF:", round(f1_score(lb, pb, average="macro"), 4))
 
    if "label2" in mine.columns:
        y2 = mine["label2"].astype(str).str.strip().str.lower().map(lab_map)
        m2 = y2.notna()
        if m2.sum() >= 20:
            kappa = cohen_kappa_score(mine.loc[m2, "y"].astype(int), y2[m2].astype(int))
            print(f"\nĐộ đồng thuận giữa hai người gán nhãn (Cohen's kappa): {kappa:.3f} trên {int(m2.sum())} câu")
            print("(kappa càng gần 1 càng đồng thuận; nếu thấp, nhãn Trung tính thường là chỗ bất đồng)")
 

In [ ]:
import gc

def train_and_eval(train_ds, val_ds, name):
    tok_tr = train_ds.map(tokenize_function, batched=True, remove_columns=["text"])
    tok_va = val_ds.map(tokenize_function, batched=True, remove_columns=["text"])

    set_seed(42)
    m = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID)

    total_steps = math.ceil(len(tok_tr) / (BATCH_SIZE * n_gpu)) * EPOCHS
    args = TrainingArguments(
        output_dir=f"./ckpt_{name}", learning_rate=2e-5,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=128,
        num_train_epochs=EPOCHS, warmup_steps=int(0.1 * total_steps),
        weight_decay=0.01, label_smoothing_factor=0.05,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="f1_macro",
        fp16=torch.cuda.is_available(), report_to="none")

    t = Trainer(model=m, args=args, train_dataset=tok_tr, eval_dataset=tok_va,
                data_collator=data_collator, compute_metrics=compute_metrics,
                callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    t.train()

    result = {}
    for test_name, test_ds in [("VSMEC", ds_social["test"]), ("VSFC", ds_review["test"])]:
        tok_te = test_ds.map(tokenize_function, batched=True, remove_columns=["text"])
        preds = np.argmax(t.predict(tok_te).predictions, axis=-1)
        result[test_name] = round(f1_score(tok_te["label"], preds, average="macro"), 4)

    del m, t
    gc.collect(); torch.cuda.empty_cache()
    return result

configs = {
    "chi_vsmec": (ds_social["train"], ds_social["validation"]),
    "chi_vsfc":  (ds_review["train"], ds_review["validation"]),
    "gop_hai":   (train_dataset, val_dataset),
}
import time
from tqdm.auto import tqdm

ket_qua = {}
t0 = time.time()
pbar = tqdm(configs.items(), total=len(configs), unit="cấu hình")

for i, (name, (tr, va)) in enumerate(pbar, start=1):
    pbar.set_description(f"Tổng tiến độ [{i}/{len(configs)}] đang train: {name}")
    t_start = time.time()

    ket_qua[name] = train_and_eval(tr, va, name)

    pbar.set_postfix_str(f"{name} xong sau {(time.time() - t_start) / 60:.1f} phút")
    print(f"✅ {name}: {ket_qua[name]}")

print(f"\n⏱ Tổng thời gian: {(time.time() - t0) / 60:.1f} phút")
print(ket_qua)

In [ ]:
tok_vsfc = ds_review["test"].map(tokenize_function, batched=True, remove_columns=["text"])
p = np.argmax(trainer.predict(tok_vsfc).predictions, axis=-1)
print(classification_report(tok_vsfc["label"], p,
      target_names=["Tiêu cực", "Trung tính", "Tích cực"], digits=4))

In [ ]:
# 1. Chọn b tốt nhất trên validation (gộp)
val_out = trainer.predict(tokenized_val)
val_logits, val_labels = val_out.predictions, val_out.label_ids

best_b, best_f1 = 0.0, -1
for b in np.arange(0, 3.01, 0.25):
    adj = val_logits.copy()
    adj[:, 1] += b                      # cột 1 = Trung tính
    f = f1_score(val_labels, adj.argmax(-1), average="macro")
    print(f"b={b:.2f} -> val f1_macro = {f:.4f}")
    if f > best_f1:
        best_b, best_f1 = b, f
print("b tốt nhất:", best_b)

# 2. So sánh trước/sau trên hai tập test
for test_name, test_ds in [("VSMEC", ds_social["test"]), ("VSFC", ds_review["test"])]:
    tok = test_ds.map(tokenize_function, batched=True, remove_columns=["text"])
    logits = trainer.predict(tok).predictions
    before = f1_score(tok["label"], logits.argmax(-1), average="macro")
    logits[:, 1] += best_b
    after = f1_score(tok["label"], logits.argmax(-1), average="macro")
    print(f"{test_name}: trước {before:.4f} -> sau {after:.4f}")